# FUTUREML02 - Churn Prediction System

## Customer Churn Prediction for Businesses

**Author:** M.V. Manikanta Reddy
**Internship:** Future Interns ML Task 02
**GitHub:** [MVManikantaReddy](https://github.com/MVManikantaReddy)

## Goal

Build a machine learning model that identifies which customers are likely to stop using a service. This notebook demonstrates data exploration, feature engineering, model training, and evaluation for churn prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Step 1: Create Synthetic Customer Churn Dataset

We'll create a realistic customer dataset with churn indicators.

In [ ]:
np.random.seed(42)
n_customers = 1000

# Generate synthetic customer data
data = {
    'CustomerID': range(1, n_customers + 1),
    'Age': np.random.randint(18, 80, n_customers),
    'Tenure_Months': np.random.randint(1, 72, n_customers),
    'Monthly_Charges': np.random.uniform(20, 120, n_customers),
    'Total_Charges': np.random.uniform(100, 8000, n_customers),
    'Contract_Type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_customers),
    'Payment_Method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_customers),
    'Internet_Service': np.random.choice(['DSL', 'Fiber optic', 'No'], n_customers),
    'Online_Security': np.random.choice(['Yes', 'No', 'No internet service'], n_customers),
    'Tech_Support': np.random.choice(['Yes', 'No', 'No internet service'], n_customers),
    'Streaming_TV': np.random.choice(['Yes', 'No', 'No internet service'], n_customers),
    'Streaming_Movies': np.random.choice(['Yes', 'No', 'No internet service'], n_customers),
    'Senior_Citizen': np.random.choice([0, 1], n_customers, p=[0.84, 0.16]),
    'Partner': np.random.choice([0, 1], n_customers, p=[0.52, 0.48]),
    'Dependents': np.random.choice([0, 1], n_customers, p=[0.70, 0.30])
}

df = pd.DataFrame(data)
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## Step 2: Add Churn Target Variable

Generate a churn label based on customer characteristics.

In [ ]:
# Generate churn probability based on features
churn_prob = (
    0.3 * (df['Tenure_Months'] < 12).astype(int) +
    0.2 * (df['Contract_Type'] == 'Month-to-month').astype(int) +
    0.15 * (df['Payment_Method'] == 'Electronic check').astype(int) +
    0.15 * (df['Internet_Service'] == 'Fiber optic').astype(int) +
    0.1 * (df['Online_Security'] == 'No').astype(int) +
    0.1 * (df['Monthly_Charges'] > 80).astype(int)
)

# Add noise and convert to binary
churn_prob = churn_prob + np.random.normal(0, 0.1, n_customers)
df['Churn'] = (churn_prob > 0.5).astype(int)

print(f'Churn rate: {df["Churn"].mean():.2%}')
print(f'Churned customers: {df["Churn"].sum()} / {n_customers}')

## Step 3: Data Exploration and Visualization

In [ ]:
# Basic data info
print('=== Dataset Info ===')
print(df.info())
print('
=== First 5 rows ===')
print(df.head())
print('
=== Statistical Summary ===')
print(df.describe())

In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn count
axes[0].pie(df['Churn'].value_counts(), labels=['Not Churned', 'Churned'], 
           autopct='%1.1f%%', colors=['#4ECDC4', '#FF6B6B'])
axes[0].set_title('Churn Distribution')

# Churn by contract type
churn_by_contract = df.groupby('Contract_Type')['Churn'].mean()
axes[1].bar(churn_by_contract.index, churn_by_contract.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1].set_title('Churn Rate by Contract Type')
axes[1].set_ylabel('Churn Rate')
axes[1].set_xticklabels(churn_by_contract.index, rotation=45, ha='right')
for i, v in enumerate(churn_by_contract.values):
    axes[1].text(i, v + 0.02, f'{v:.2%}', ha='center')

plt.tight_layout()
plt.show()

## Step 4: Feature Engineering and Preprocessing

In [ ]:
# Encode categorical variables
le = LabelEncoder()
categorical_cols = ['Contract_Type', 'Payment_Method', 'Internet_Service', 'Online_Security', 
                   'Tech_Support', 'Streaming_TV', 'Streaming_Movies']

for col in categorical_cols:
    df[col + '_Encoded'] = le.fit_transform(df[col])

# Select features for modeling
feature_cols = ['Age', 'Tenure_Months', 'Monthly_Charges', 'Total_Charges', 
               'Senior_Citizen', 'Partner', 'Dependents',
               'Contract_Type_Encoded', 'Payment_Method_Encoded', 'Internet_Service_Encoded',
               'Online_Security_Encoded', 'Tech_Support_Encoded', 
               'Streaming_TV_Encoded', 'Streaming_Movies_Encoded']

X = df[feature_cols]
y = df['Churn']

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Train churn rate: {y_train.mean():.2%}')
print(f'Test churn rate: {y_test.mean():.2%}')

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Features scaled successfully!')

## Step 5: Model Training and Evaluation

In [ ]:
# Define evaluation function
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f'\n{model_name} Results:')
    print(f' Accuracy: {acc:.4f}')
    print(f' Precision: {prec:.4f}')
    print(f' Recall: {rec:.4f}')
    print(f' F1-Score: {f1:.4f}')
    print(f' ROC-AUC: {roc_auc:.4f}')
    
    cm = confusion_matrix(y_test, y_pred)
    print(f' Confusion Matrix:\n{cm}')
    
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'roc_auc': roc_auc}

In [ ]:
# Train Logistic Regression
print('='*60)
print('Training Models...')
print('='*60)

print('\n- Logistic Regression (Baseline)')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_results = evaluate_model(lr_model, X_test_scaled, y_test, 'Logistic Regression')

In [ ]:
# Train Random Forest
print('\n- Random Forest Classifier')
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
rf_results = evaluate_model(rf_model, X_test, y_test, 'Random Forest')

In [ ]:
# Train XGBoost
print('\n- XGBoost Classifier')
xgb_model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_results = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

## Step 6: Model Performance Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models_results = {'Logistic Regression': lr_results, 'Random Forest': rf_results, 'XGBoost': xgb_results}
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

for idx, metric in enumerate(metrics[:3]):
    values = [models_results[model][metric] for model in models_results.keys()]
    axes[idx].bar(models_results.keys(), values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    axes[idx].set_ylabel(metric.capitalize())
    axes[idx].set_title(f'{metric.upper()} Comparison')
    axes[idx].set_ylim([0, 1])
    axes[idx].set_xticklabels(models_results.keys(), rotation=45, ha='right')
    for i, v in enumerate(values):
        axes[idx].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from Random Forest
fig, ax = plt.subplots(figsize=(10, 6))
importances = rf_model.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[::-1]

ax.barh(range(len(importances)), importances[indices], color='#4ECDC4')
ax.set_yticks(range(len(importances)))
ax.set_yticklabels([feature_names[i] for i in indices])
ax.set_xlabel('Feature Importance')
ax.set_title('Top Features for Churn Prediction (Random Forest)')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we built a churn prediction system using three machine learning models:

1. **Logistic Regression** - Serves as our baseline model
2. **Random Forest** - An ensemble method that captures non-linear relationships
3. **XGBoost** - A gradient boosting algorithm known for high performance

Key findings:
- Customers with shorter tenure are more likely to churn
- Month-to-month contracts have higher churn rates
- Electronic check payment method correlates with higher churn
- Fiber optic internet service customers show higher churn tendency

**Author:** M.V. Manikanta Reddy
**GitHub:** [MVManikantaReddy](https://github.com/MVManikantaReddy)